In [ ]:
from elasticsearch import elasticsearch
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta

es = Elasticsearch(['http://elasticsearch:9200'])

In [ ]:
# Запрос данных за последние 24 часа
res = es.search(
    index='logs-*',
    body={
        "query": {
            "range": {
                "@timestamp": {
                    "gte": "now-id"
                }
            }
        }
        "size": 10000,
        "sort": [{"@timestamp": "desc"}]
    }
)

# Преобразование в DataFrame
logs = []
for hit in res['hits']['hits']:
    log = hit['_source']
    log['_id'] = hit['_id']
    logs.append(log)

df = pd.json_normalize(logs)
df['@timestamp'] = pd.to_datetime(df['@timestamp'])

# Анализ ошибок по часам
df['hour'] = df['@timestamp'].dt.hour
errors_by_hour = df[df['log_level'] == 'ERROR'].groupby('hour').size()

plt.figure(figsize=(12, 6))
errors_by_hour.plot(kind='bar')
plt.title('Распределение ошибок по часам')
plt.xlabel('Час')
plt.ylabel('Количество ошибок')
plt.show()

# Топ источников ошибок
top_sources = df[df['log_level'] == 'ERROR']['source'].value_counts().head(10)
plt.figure(figsize=(10, 6))
top_sources.plot(kind='barh')
plt.title('Топ-10 источников ошибок')
plt.xlabel('Количество ошибок')
plt.show()

# Анализ latency (если есть)
if 'response_time' in df.columns:
    df['response_time'] = pd.to_numeric(df['response_time'], errors='coerce')

    plt.figure(figsize=(12, 6))
    plt.plot(df['@timestamp'], df['response_time'], 'o', alpha=0.5)
    plt.title('Response time over time')
    plt.xlabel('Time')
    plt.ylabel('Response time (ms)')
    plt.show()